# Levanter (fixed) — Colab training + HF export

This notebook sets up a **working** Levanter training run on Colab (GPU), following the approach that worked in your second snippet.

**What’s fixed vs the broken version:**
- Installs Levanter with deps (so `tqdm_loggable` et al. are present)
- Pins `equinox<0.13,!=0.12.0`
- Aligns `jax==0.7.2`, `jaxlib==0.7.2`, and `jax-cuda12-plugin==0.7.2`
- Pins `protobuf>=6,<7` to match Levanter/TB/Ray (use uv below if Colab conflicts)
- Uses `--trainer.tracker '{type: noop}'` and disables JAXPR/HLO logs
- GPT‑2 *sampling* is done via **HF export** (Levanter’s `sample_lm.py` targets LLaMA)

You can also optionally train **LLaMA-small** and sample with Levanter’s built‑in script.

In [ ]:
# 0) Mount Google Drive (for persistent repo/cache/checkpoints)
try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
except Exception as e:
  print("Not in Colab or drive mount failed; continuing without Drive.")

import os, pathlib
PROJECT_ROOT = "/content/drive/MyDrive/colab_projects/levanter"
project_path = pathlib.Path(PROJECT_ROOT)
project_path.parent.mkdir(parents=True, exist_ok=True)
print("Using PROJECT_ROOT:", PROJECT_ROOT)
if project_path.exists():
  print("PROJECT_ROOT already exists; the git-sync cell will reuse it.")
else:
  print("PROJECT_ROOT will be created when the git-sync cell runs.")


## 1) Clone / sync the repo (your branch)
If your feature branch isn’t available, change `BRANCH` to `main`.

In [ ]:
import os, pathlib, shutil, subprocess, sys, tempfile
REPO_URL = "https://github.com/chris544460/levanter.git"
BRANCH = "feat/style-prefix-token"  # set to 'main' if needed

PROJECT_PATH = pathlib.Path(PROJECT_ROOT)
PROJECT_PATH.parent.mkdir(parents=True, exist_ok=True)
GIT_DIR = PROJECT_PATH / ".git"
PERSIST_DIRS = {"cache", "checkpoints"}
IGNORABLE_ENTRIES = {".ipynb_checkpoints", ".DS_Store"}

def run(cmd):
  print("$", cmd)
  status = subprocess.call(cmd, shell=True)
  if status != 0:
    raise SystemExit(f"Command failed: {cmd}")

def stash_preservable_entries():
  preserved = []
  leftovers = []
  if not PROJECT_PATH.exists():
    return preserved, leftovers
  for entry in PROJECT_PATH.iterdir():
    if entry.name in PERSIST_DIRS and entry.is_dir():
      tmp_dir = pathlib.Path(tempfile.mkdtemp(prefix=f"levanter-preserve-{entry.name}-", dir=PROJECT_PATH.parent))
      dest = tmp_dir / entry.name
      shutil.move(str(entry), dest)
      preserved.append((entry.name, dest, tmp_dir))
    elif entry.name in IGNORABLE_ENTRIES:
      if entry.is_dir():
        shutil.rmtree(entry)
      else:
        try:
          entry.unlink()
        except FileNotFoundError:
          pass
    else:
      leftovers.append(entry.name)
  return preserved, leftovers

def restore_preserved(preserved):
  for name, saved_path, tmp_dir in preserved:
    target = PROJECT_PATH / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(saved_path), str(target))
    shutil.rmtree(tmp_dir, ignore_errors=True)

if not GIT_DIR.exists():
  preserved, leftovers = stash_preservable_entries()
  if leftovers:
    raise SystemExit(
      "PROJECT_ROOT already contains files that are not part of a repo: "
      + ", ".join(leftovers)
      + ". Remove them or point PROJECT_ROOT to an empty directory."
    )
  if PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)
  PROJECT_PATH.mkdir(parents=True, exist_ok=True)
  try:
    run(f"git clone -b {BRANCH} --single-branch {REPO_URL} {PROJECT_ROOT}")
  except SystemExit:
    restore_preserved(preserved)
    raise
  restore_preserved(preserved)
else:
  run(f"git -C {PROJECT_ROOT} fetch origin {BRANCH}")
  run(f"git -C {PROJECT_ROOT} checkout {BRANCH} || true")
  run(f"git -C {PROJECT_ROOT} reset --hard origin/{BRANCH}")
  run(f"git -C {PROJECT_ROOT} clean -fd")

CACHE_ROOT = os.path.join(PROJECT_ROOT, "cache")
pathlib.Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["CACHE_ROOT"] = CACHE_ROOT
print("Using CACHE_ROOT:", CACHE_ROOT)

!git -C "$PROJECT_ROOT" rev-parse --abbrev-ref HEAD
!git -C "$PROJECT_ROOT" log -1 --oneline


## 2) Install Levanter (editable) + **compatibility pins**
We:
1. Upgrade pip tooling and bump `draccus>=0.11.5` up front so `--config_path` is recognized.
2. Pin `protobuf>=6,<7` to match Levanter/TensorBoard/Ray. If Colab conflicts, use the uv path below.
3. Install Levanter in editable mode **with** its dependencies (fixes missing modules like `tqdm_loggable`).
4. Pin `equinox<0.13,!=0.12.0` (Levanter still uses older APIs).
5. Ensure `jax[cuda12]==0.7.2` to match `jaxlib` and the CUDA12 PJRT plugin.
6. Re-run the editable install after pinning so Colab keeps loading the repo copy.


In [ ]:
%cd $PROJECT_ROOT
!python -m pip install -U pip wheel setuptools

# Upgrade draccus before installing Levanter so --config_path is accepted.
!python -m pip install -U "draccus>=0.11.5"

# Pin protobuf alongside draccus to avoid Ray/TensorBoard conflicts.
!python -m pip install -U "protobuf>=6,<7"

# Install Levanter with dependencies (editable)
!python -m pip install -e .

# Pin Equinox (>=0.11.x, but <0.13 and not 0.12.0) to avoid API breakage
!python -m pip install "equinox<0.13,!=0.12.0"

# Ensure JAX + CUDA12 plugin match (prevents PJRT aborts)
!python -m pip install -U "jax[cuda12]==0.7.2"

# Re-install Levanter after dependency pinning to keep the editable install active
!python -m pip install -e .


## 3) Verify imports and devices
If you still see a device mismatch warning, restart the runtime and re‑run from the top.

In [ ]:
# No-op: keep working directory at $PROJECT_ROOT
cd ..

In [ ]:
import jax, levanter, importlib, os, re
print("JAX version:", jax.__version__)
try:
  import jax_cuda12_plugin as _jcp
  print("jax-cuda12-plugin:", getattr(_jcp, "__version__", "present"))
except Exception as e:
  print("jax-cuda12-plugin not importable (OK on CPU runtimes)")
print("JAX devices:", jax.devices())
print("Levanter version:", getattr(levanter, "__version__", "dev"))
import google.protobuf, draccus
print("protobuf version:", getattr(google.protobuf, "__version__", "unknown"))
print("draccus version:", getattr(draccus, "__version__", "unknown"))

def _parse_semver(value: str) -> tuple[int, int, int]:
  parts = [int(x) for x in re.findall(r"\d+", value)]
  while len(parts) < 3:
    parts.append(0)
  return tuple(parts[:3])

if _parse_semver(getattr(draccus, "__version__", "0")) < (0, 11, 5):
  raise SystemExit(
      f"draccus {draccus.__version__} is too old for --config_path. "
      "Run `pip install -U \"draccus>=0.11.5\"` and rerun the install cell."
  )


## 4) Train GPT‑2 “nano” (toy demo)
This follows the README quickstart. It runs for ~100 steps on WikiText‑103. We disable W&B and heavy debug logs.
> We call the CLI (`python -m levanter.main.train_lm`) to avoid `asyncio.run` clashes with the Colab event loop.


In [ ]:
# Run via CLI to avoid asyncio.run conflicts in Colab
%cd $PROJECT_ROOT
# Refresh editable install so the real draccus package is available in this fresh process
!python -m pip install -e . --quiet
%cd $PROJECT_ROOT/src
!python -m levanter.main.train_lm \
  --config_path '/content/drive/MyDrive/colab_projects/levanter/config/gpt2_nano.yaml' \
  --trainer.tracker '{type: noop}' \
  --trainer.log_jaxprs false \
  --trainer.log_xla_hlo false


## 5) Export the latest checkpoint to Hugging Face format (for GPT‑2) and sample
Levanter’s built‑in sampler targets LLaMA; for GPT‑2 we export to HF and use `transformers` to generate.

In [ ]:
import os, glob, json
%cd $PROJECT_ROOT
cands = [p for p in glob.glob(os.path.join('checkpoints', '*', 'step-*'))
         if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
  try:
    with open(os.path.join(p, 'metadata.json')) as f:
      m = json.load(f)
    return (m.get('timestamp', ''), int(m.get('step', -1)))
  except Exception:
    return ('', -1)
if not cands:
  raise SystemExit('No checkpoints found under checkpoints/. Run the training cell above first.')
LATEST = sorted(cands, key=score)[-1]
print('Latest checkpoint =>', LATEST)

In [ ]:
%cd $PROJECT_ROOT
!python -m levanter.main.export_lm_to_hf \
  --checkpoint_path "$LATEST" \
  --output_dir /tmp/gpt2_nano_hf \
  --model.type gpt2 \
  --model.hidden_dim 32 \
  --model.num_layers 2 \
  --model.num_heads 4

In [ ]:
# Sample with transformers (CPU is fine for this tiny model)
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("/tmp/gpt2_nano_hf")
model = AutoModelForCausalLM.from_pretrained("/tmp/gpt2_nano_hf")
prompt = "Question: What's the capital of Germany?\nAnswer:"
inputs = tok(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=32)
print(tok.decode(outputs[0], skip_special_tokens=True))
print("\n(Expect random-ish text; this tiny toy model doesn’t learn facts.)")

## 6) (Optional) Train LLaMA‑small on OpenWebText
This one builds caches into Drive and trains a small LLaMA config. You can later sample with Levanter’s `sample_lm.py`.

> If HF requires auth for your tokenizer, set `huggingface-cli login` in a separate cell.
> Same CLI approach keeps Colab's event loop happy during checkpointing.


In [ ]:
# Run via CLI to avoid asyncio.run conflicts in Colab
%cd $PROJECT_ROOT
# Refresh editable install so the real draccus package is available in this fresh process
!python -m pip install -e . --quiet
%cd $PROJECT_ROOT/src
!python -m levanter.main.train_lm \
  --config_path 'config/llama_small_fast.yaml' \
  --data.cache_dir "${CACHE_ROOT}/openwebtext" \
  --trainer.tracker '{type: noop}' \
  --trainer.log_jaxprs false \
  --trainer.log_xla_hlo false


### Sample the most recent LLaMA‑small checkpoint (uses Levanter’s sampler)
Update the model dims if you edited the config.

In [ ]:
%cd $PROJECT_ROOT
import os, glob, json
cands = [p for p in glob.glob(os.path.join('checkpoints', '*', 'step-*'))
         if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
  try:
    with open(os.path.join(p, 'metadata.json')) as f:
      m = json.load(f)
    return (m.get('timestamp', ''), int(m.get('step', -1)))
  except Exception:
    return ('', -1)
if not cands:
  raise SystemExit('No checkpoints found. Train LLaMA-small first.')
LATEST_LLAMA = sorted(cands, key=score)[-1]
print('Latest checkpoint =>', LATEST_LLAMA)

!python -m levanter.main.sample_lm \
  --checkpoint_path "$LATEST_LLAMA" \
  --tokenizer NousResearch/Llama-2-7b-hf \
  --model.type llama \
  --model.hidden_dim 768 \
  --model.intermediate_dim 2048 \
  --model.num_heads 12 \
  --model.num_kv_heads 12 \
  --model.num_layers 12 \
  --model.seq_len 1024 \
  --temperature 0.0 \
  --max_new_tokens 16 \
  --prompts "What is the capital of France?"

---
## (Optional) Use an isolated **uv** environment
If you want Levanter to keep `protobuf>=6` while Colab stays on `<6`, you can run via `uv` (it creates a `.venv` under the repo):

In [ ]:
%cd $PROJECT_ROOT
!python -m pip install -U uv

# Install project + pins inside the uv venv
!uv pip install -e .
!uv pip install "equinox<0.13,!=0.12.0" "jax[cuda12]==0.7.2" "protobuf>=6,<7" "draccus>=0.11.5"

In [ ]:
# Run via CLI to avoid asyncio.run conflicts in Colab
%cd $PROJECT_ROOT/src
!uv run python -m levanter.main.train_lm \
  --config_path 'config/gpt2_nano.yaml' \
  --trainer.tracker '{type: noop}' \
  --trainer.log_jaxprs false \
  --trainer.log_xla_hlo false


## Notes & gotchas
- **GPU plugin mismatch**: If you see PJRT errors mentioning `abort_collectives_on_failure` or “plugin version not compatible”, re‑run the JAX install cell (`jax[cuda12]==0.7.2`).
- **Equinox API**: Keep `equinox<0.13,!=0.12.0` unless the repo updates its calls.
- **Protobuf**: Levanter expects `protobuf>=6,<7`. If Colab's global env conflicts, prefer the **uv** path above for isolation.
- **Sampling GPT‑2**: Use the HF export route (Levanter’s `sample_lm.py` checks for LLaMA).
- **W&B**: The training cells use `{type: noop}`. If you want W&B, run `wandb login` and drop the tracker override.
- **Persistence**: Checkpoints and caches live under Drive (`PROJECT_ROOT`) so you don’t lose them when the Colab VM resets.